In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab10.ipynb")

# Lab 10: Correlation and Linear Regression
**DATA2201 — Foundations of Data Science**  
**9 questions | 100 points | Fully autograded**

Apply Lectures 18–19 to real chemical measurements of wines: standard units,
correlation, nearby-value predictions, and linear regression.

**Reading:** [Correlation](https://inferentialthinking.com/chapters/15/1/Correlation.html)
and [The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html).

## Before You Start
- Use population standard deviations: `np.std(values, ddof=0)`.
- Use short intermediate steps rather than long expressions. Do not round answers.
- Do not use built-in correlation or regression methods in your answers:
  `.corr()`, `np.corrcoef`, `np.polyfit`, or regression libraries.
- Keep the original `wine` DataFrame unchanged and preserve row order.
- Public checks are examples. Hidden tests also check numerical correctness.
- Work with classmates and submit jointly as instructed.

---
<a id="outline"></a>

## Lab09 Question Navigation

1. [Standard Units](#q1)
2. [Correlation and Measurement Units](#q2)
3. [Association Within Cultivars](#q3)
4. [Nearby-Value Predictions](#q4)
5. [Regression Parameters](#q5)
6. [Predictions in Original Units](#q6)
7. [Switching Predictor and Response](#q7)
8. [Regression in Standard Units](#q8)
9. [Interpreting the Results](#q9)

## Required Packages

In [ ]:
# Run this setup cell; do not change it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')

## Real Dataset: Wine Chemical Measurements
The supplied **`wine.csv`** contains **178 wines** from three cultivars grown in
one region of Italy. Each row is one wine; the measurements are real observations.

Source: [UCI Wine dataset](https://archive.ics.uci.edu/dataset/109/wine),
credited to S. Aeberhard and M. Forina. This CSV uses scikit-learn's bundled copy
and adds `cultivar` labels 1, 2, and 3 corresponding to its three classes.

The file includes all 13 chemical measurements and `cultivar`. We focus on:

| Column | Meaning |
| :-- | :-- |
| `alcohol` | Alcohol measurement, in percent by volume |
| `color_intensity` | Numeric measure of color intensity, in dataset units |
| `cultivar` | Cultivar identifier: 1, 2, or 3 |

We will predict **color intensity from alcohol**. The dataset has no missing values.
No installation of scikit-learn or network access is needed to run this lab.

## Load the dataset

In [ ]:
wine = pd.read_csv('data/wine.csv')
wine.head()

In [ ]:
# Provided plot: inspect the overall pattern and the cultivar groups.
fig, ax = plt.subplots(figsize=(7, 4))

for cultivar, group in wine.groupby('cultivar'):
    ax.scatter(
        group['alcohol'],
        group['color_intensity'],
        label=f'Cultivar {cultivar}',
        s=25
    )

ax.set_xlabel('Alcohol (% by volume)')
ax.set_ylabel('Color intensity (dataset units)')
ax.set_title('Alcohol and Color Intensity')
ax.legend()
plt.tight_layout()
plt.show()

<a id="q1"></a>

### Question 1: Standard Units (10 points)


Convert the alcohol and color columns to standard units:

1. Create NumPy arrays named `alcohol_values` and `color_values` from the corresponding columns.
2. Calculate each array's **mean** and **population standard deviation**. Store the results in `alcohol_mean`, `alcohol_sd`, `color_mean`, and `color_sd`.
3. Use these intermediate variables to create standardized arrays named `alcohol_su` and `color_su`:

   $$
   z_i = \frac{x_i - \operatorname{mean}(x)}{\operatorname{std}(x)}
   $$

Keep the original row order. Use the intermediate variables to keep each step short.

In [ ]:
alcohol_values = ...
color_values = ...
alcohol_mean = ...
alcohol_sd = ...
color_mean = ...
color_sd = ...
alcohol_su = ...
color_su = ...

In [ ]:
grader.check("q1")

[↑ Return to Lab Outline](#outline)

<a id="q2"></a>
### Question 2: Correlation and Measurement Units (12 points)


1. Assign `su_products` to the elementwise products of the two standard-unit arrays.
2. Assign `alcohol_color_r` to the mean of those products.
3. Convert alcohol from percentage points to a proportion by dividing its values
   by 100. Store this array as `alcohol_fraction`.
4. Standardize the converted values as `fraction_su`, then compute `fraction_r`
   as the mean of their products with `color_su`.

This comparison investigates whether changing units affects correlation.

In [ ]:
su_products = ...
alcohol_color_r = ...
alcohol_fraction = ...
fraction_mean = ...
fraction_sd = ...
fraction_su = ...
fraction_r = ...

In [ ]:
grader.check("q2")

[↑ Return to Lab Outline](#outline)

<a id="q3"></a>
### Question 3: Association Within Cultivars (10 points)


Calculate the alcohol/color-intensity correlation **within each cultivar**.
Store the results in a pandas Series `cultivar_r`, indexed in order by `1`, `2`, `3`.

For each group, standardize both measurements using that group's own mean and
population standard deviation, then average their paired products. A loop is
allowed; a loop outline is provided. Do not define a function.

Assign `positive_cultivar_count` to the number of positive group correlations.
Assign `strongest_cultivar` to the cultivar whose correlation has the largest
**absolute value**. Hint: use `.abs().idxmax()`.

In [ ]:
cultivar_r = pd.Series(index=[1, 2, 3], dtype=float)

for label in cultivar_r.index:
    group = ...
    group_x = ...
    group_y = ...
    group_x_su = ...
    group_y_su = ...
    cultivar_r.loc[label] = ...

positive_cultivar_count = ...
strongest_cultivar = ...

In [ ]:
grader.check("q3")

[↑ Return to Lab Outline](#outline)

<a id="q4"></a>
### Question 4: Nearby-Value Predictions (12 points)


Use the lecture's nearby-value approach to predict color intensity at **13% alcohol**.

- `nearby_wines`: all original columns for rows with **12.75 ≤ alcohol < 13.25**.
- `nearby_count`: the number of selected wines.
- `nearby_prediction`: the mean color intensity of those wines.
- `wide_wines`: rows with **12.5 ≤ alcohol < 13.5**.
- `wide_prediction`: the mean color intensity in the wider window.

Use separate lower/upper Boolean conditions for readability. Retain original row
order and index. Both windows include the lower endpoint and exclude the upper.

In [ ]:
near_lower = ...
near_upper = ...
nearby_wines = ...
nearby_count = ...
nearby_prediction = ...
wide_lower = ...
wide_upper = ...
wide_wines = ...
wide_prediction = ...

In [ ]:
grader.check("q4")

[↑ Return to Lab Outline](#outline)

<a id="q5"></a>
### Question 5: Regression Parameters (14 points)


Fit the regression predicting **color intensity from alcohol** using all wines.
Assign `model_slope` and `model_intercept` using your earlier summary variables:
$$\text{slope}=r\frac{\operatorname{std}(y)}{\operatorname{std}(x)},\qquad
\text{intercept}=\operatorname{mean}(y)-\text{slope}\operatorname{mean}(x).$$

Create `model_parameters`, a NumPy array in `[r, slope, intercept]` order.
Use direct expressions and the lecture formulas.

In [ ]:
model_slope = ...
model_intercept = ...
model_parameters = ...

In [ ]:
grader.check("q5")

[↑ Return to Lab Outline](#outline)

<a id="q6"></a>
### Question 6: Predictions in Original Units (12 points)


Apply $\widehat y=\text{slope}\,x+\text{intercept}$.

1. Set `new_alcohol = np.array([12.0, 13.0, 14.0])` and calculate the NumPy array
   `new_predictions` for these measurements.
2. Make a copy of the original data named `wine_predictions`. Add
   `linear_prediction` for every wine.
3. Add the Boolean column `above_prediction`, indicating whether observed color
   intensity is **strictly greater than** the corresponding prediction.

Do not change `wine`. These are pooled predictions across all cultivars.

In [ ]:
new_alcohol = np.array([12.0, 13.0, 14.0])
new_predictions = ...
wine_predictions = ...
observed_alcohol = ...
wine_predictions['linear_prediction'] = ...
observed_color = ...
fitted_color = ...
wine_predictions['above_prediction'] = ...

In [ ]:
grader.check("q6")

[↑ Return to Lab Outline](#outline)

<a id="q7"></a>
### Question 7: Switching Predictor and Response (10 points)


Reverse the model: predict **alcohol from color intensity**.

Compute `reverse_slope` and `reverse_intercept` using the lecture formulas.
Assign `slope_product` to the product of the forward and reverse slopes.

Create a Boolean NumPy array `reverse_claims` for these three statements, in order:
1. Switching predictor and response leaves the correlation unchanged.
2. The reverse slope equals the reciprocal of the forward slope.
3. The product of the two slopes equals $r^2$.

Use the fitted results to decide whether each statement is true for this dataset.

In [ ]:
reverse_slope = ...
reverse_intercept = ...
slope_product = ...
reverse_claims = ...

In [ ]:
grader.check("q7")

[↑ Return to Lab Outline](#outline)

<a id="q8"></a>
### Question 8: Regression in Standard Units (10 points)


Use the equation $\widehat y_{SU}=r\,x_{SU}$ for the values in `new_alcohol`.

- `new_x_su`: standardize using the **full observed alcohol dataset's** mean and
  population standard deviation, not the three new values' statistics.
- `predicted_y_su`: predictions in standard units.
- `predictions_from_su`: convert the predictions back to color-intensity units
  using the **full observed color dataset's** mean and population standard deviation.

Each result must be a NumPy array. Use the standard-unit equations, rather than
the fitted slope and intercept. The final predictions should agree with Q6.

In [ ]:
new_x_su = ...
predicted_y_su = ...
predictions_from_su = ...

In [ ]:
grader.check("q8")

[↑ Return to Lab Outline](#outline)

<a id="q9"></a>
### Question 9: Interpreting the Results (10 points)


Assign `interpretation_answers` to a NumPy array of **five integer option numbers**,
one per item, in the order A–E.

**A.** The observed positive correlation proves that increasing alcohol causes
increased color intensity.  
`1`: True. `2`: False; correlation alone does not establish causation.

**B.** The pooled alcohol/color correlation is stronger than the correlations
within individual cultivars.  
`1`: The pooled correlation must apply to each cultivar.
`2`: Group composition can affect a pooled relationship; examine groups separately.

**C.** A symmetric U-shaped relationship has correlation near zero.  
`1`: A strong nonlinear association can still exist.
`2`: There cannot be any association.

**D.** Someone uses our model at **20% alcohol**, outside the observed range.  
`1`: This is interpolation and is reliable.
`2`: This is extrapolation; the data do not establish that the trend continues there.

**E.** For an alcohol value one standard deviation above its observed mean,
the fitted color-intensity prediction in standard units is:  
`1`: Exactly one standard deviation above the color-intensity mean.
`2`: $r$ standard deviations above the color-intensity mean.

Example format only: `np.array([1, 1, 2, 1, 1])`. Choose your own answers.

In [ ]:
interpretation_answers = ...

In [ ]:
grader.check("q9")

[↑ Return to Lab Outline](#outline)

<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Congratulations! You Have Finished Lab 10!
Restart the kernel, run all cells, check the public tests, and save your notebook.
Use the export cell generated by Otter to prepare the required submission.
The lab has **100 autograded points** and no manual grading.

[↑ Return to Lab Outline](#outline)

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)